In [2]:
# define State

# === Main State ===
import os
from typing import TypedDict

from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, AIMessage, SystemMessage
from langgraph.graph import END, START, StateGraph


class TravelPlanState(TypedDict):
    # user input
    destination: str
    start_date: str
    end_date: str
    budget: float
    num_travelers: int

    # Agent Professional result
    flight_recommendations: list
    hotel_recommendations: list
    activity_recommendations: list
    budget_breakdown: dict

    # final plan
    final_plan: str
    total_cost: float

# === flight SubGraph ===
class FlightSearchState(TypedDict):
    destination: str
    start_date: str
    end_date: str
    num_travelers: int
    search_results: list
    recommendations: list

# === hotel SubGraph ===
class HotelSearchState(TypedDict):
    destination: str
    start_date: str
    end_date: str
    num_travelers: int
    search_results: list
    recommendations: list

# === Activity SubGraph ===
class ActivitySearchState(TypedDict):
    destination: str
    start_date: str
    end_date: str
    num_travelers: int
    search_results: list
    recommendations: list


# define model
model = init_chat_model(
    model="gpt-4o-mini",
    model_provider="openai",
    temperature=0.0,
    base_url = os.getenv("base_url"),
    api_key = os.getenv("api_key"),
    max_tokens=None,
    max_retries=2,
)

# ==============================================================
# tools and functions
# ==============================================================

def search_flights_api(destination: str, date: str, travelers: int) -> list:
    """模拟航班搜索 API"""
    # 实际应用中调用真实的航班 API
    return [
        {
            "airline": "航空公司 A",
            "departure": "08:00",
            "arrival": "12:00",
            "price": 1200,
            "duration": "4h"
        },
        {
            "airline": "航空公司 B",
            "departure": "14:00",
            "arrival": "18:00",
            "price": 950,
            "duration": "4h"
        }
    ]

def search_hotels_api(destination: str, checkin: str, checkout: str) -> list:
    """模拟酒店搜索 API"""
    return [
        {
            "name": "豪华酒店",
            "rating": 5,
            "price_per_night": 800,
            "amenities": ["游泳池", "健身房", "早餐"]
        },
        {
            "name": "经济酒店",
            "rating": 3,
            "price_per_night": 300,
            "amenities": ["WiFi", "早餐"]
        }
    ]

def search_activities_api(destination: str) -> list:
    """模拟活动搜索 API"""
    return [
        {"name": "城市观光", "duration": "4h", "price": 200},
        {"name": "博物馆参观", "duration": "3h", "price": 150},
        {"name": "美食tour", "duration": "3h", "price": 300}
    ]

# ==============================================================
# 子图 1：航班专家
# ==============================================================

def flight_search_node(state: FlightSearchState) -> dict:
    """搜索航班"""
    print("✈️  搜索航班...")

    results = search_flights_api(
        state["destination"],
        state["start_date"],
        state["num_travelers"]
    )

    return {"search_results": results}

def flight_recommend_node(state: FlightSearchState) -> dict:
    """recommend flight"""
    import json
    print("✈️  AI 分析航班...")

    results = state["search_results"]

    # 使用 LLM 分析
    prompt = f"""\
基于以下航班信息，推荐最佳选项:
{json.dumps(results, ensure_ascii=False, indent=2)}

返回推荐理由。
"""

    response = model.invoke([HumanMessage(content=prompt)])

    recommendations = [
        {
            "flight": results[1],   # 选择经济型
            "reason": response.content
        }
    ]
    return {"recommendations": recommendations}

def hotel_search_node(state: HotelSearchState) -> dict:
    """搜索酒店"""
    print("🏨  搜索酒店...")
    results = search_hotels_api(
        state["destination"],
        state["start_date"],
        state["end_date"]
    )
    return {"search_results": results}

def hotel_recommend_node(state: HotelSearchState) -> dict:
    import json
    """AI 推荐酒店"""
    print("🏨  AI 分析酒店...")

    results = state["search_results"]

    # 使用 LLM 分析
    prompt = f"""\
基于以下酒店信息，推荐最佳选项:
{json.dumps(results, ensure_ascii=False, indent=2)}

返回推荐理由。
"""

    response = model.invoke([HumanMessage(content=prompt)])

    recommendations = [
        {
            "hotel": results[1],   # 选择经济型
            "reason": response.content
        }
    ]
    return {"recommendations": recommendations}

def activity_search_node(state: ActivitySearchState) -> dict:
    """搜索活动"""
    print("🎉  搜索活动...")
    results = search_activities_api(state["destination"])
    return {"search_results": results}

def activity_recommend_node(state: ActivitySearchState) -> dict:
    import json
    """AI 推荐活动"""
    print("🎉  AI 分析活动...")

    results = state["search_results"]

    # 使用 LLM 分析
    prompt = f"""\
基于以下活动信息，推荐最佳选项:
{json.dumps(results, ensure_ascii=False, indent=2)}

返回推荐理由。
"""
    response = model.invoke([HumanMessage(content=prompt)])

    recommendations = [
        {
            "activity": results[0],  # 选择第一个活动作为推荐
            "reason": response.content
        }
    ]
    return {"recommendations": recommendations}


# create flight graph
flight_graph = StateGraph(FlightSearchState)
flight_graph.add_node("search", flight_search_node)
flight_graph.add_node("recommend", flight_recommend_node)

flight_graph.add_edge(START, "search")
flight_graph.add_edge("search", "recommend")
flight_graph.add_edge("recommend", END)

# compile flight_graph
flight_agent = flight_graph.compile()

#create hotel graph
hotel_graph = StateGraph(HotelSearchState)
hotel_graph.add_node("search", hotel_search_node)
hotel_graph.add_node("recommend", hotel_recommend_node)

hotel_graph.add_edge(START, "search")
hotel_graph.add_edge("search", "recommend")
hotel_graph.add_edge("recommend", END)

# compile hotel_graph
hotel_agent = hotel_graph.compile()

#create activity graph
activity_graph = StateGraph(ActivitySearchState)
activity_graph.add_node("search", activity_search_node)
activity_graph.add_node("recommend", activity_recommend_node)

activity_graph.add_edge(START, "search")
activity_graph.add_edge("search", "recommend")
activity_graph.add_edge("recommend", END)

# compile activity_graph
activity_agent = activity_graph.compile()


# ===========================================================
# Main Agent Design
# ===========================================================


# 封装所有专家Agent为一个Node:
def call_flight_expert(state: TravelPlanState) -> dict:
    """调用航班专家"""
    result = flight_agent.invoke({
        "destination": state["destination"],
        "start_date": state["start_date"],
        "end_date": state["end_date"],
        "num_travelers": state["num_travelers"],
        "search_results": [],
        "recommendations": []
    })
    return {"flight_recommendations": result["recommendations"]}

def call_hotel_expert(state: TravelPlanState) -> dict:
    """调用酒店专家"""
    result = hotel_agent.invoke({
        "destination": state["destination"],
        "start_date": state["start_date"],
        "end_date": state["end_date"],
        "num_travelers": state["num_travelers"],
        "search_results": [],
        "recommendations": []
    })
    return {"hotel_recommendations": result["recommendations"]}

def call_activity_expert(state: TravelPlanState) -> dict:
    """调用活动专家"""
    result = activity_agent.invoke({
        "destination": state["destination"],
        "start_date": state["start_date"],
        "end_date": state["end_date"],
        "num_travelers": state["num_travelers"],
        "search_results": [],
        "recommendations": []
    })
    return {"activity_recommendations": result["recommendations"]}

def calculate_budget(state: TravelPlanState) -> dict:
    """计算预算 - 修复版本"""
    print("💰  计算预算...")
    
    # 检查是否所有专家推荐都已完成
    flight_recs = state.get("flight_recommendations", [])
    hotel_recs = state.get("hotel_recommendations", [])
    activity_recs = state.get("activity_recommendations", [])
    
    if not all([flight_recs, hotel_recs, activity_recs]):
        print("⏳  等待所有专家完成推荐...")
        return {}  # 数据不完整，不执行计算
    
    # 检查是否已经计算过预算
    if state.get("budget_breakdown"):
        print("✅  预算已计算完成，跳过重复计算")
        return {}  # 已经计算过了，避免重复
    
    print("🔢  开始计算预算...")
    
    # 航班费用 - 修正数据访问路径
    flight_cost = 0
    if flight_recs and len(flight_recs) > 0:
        flight_data = flight_recs[0].get("flight", {})
        if "price" in flight_data:
            flight_cost = flight_data["price"] * state["num_travelers"]
            print(f"✈️  航班费用: {flight_data['price']} x {state['num_travelers']} = {flight_cost}")

    # 酒店费用 - 修正数据访问路径
    hotel_cost = 0
    if hotel_recs and len(hotel_recs) > 0:
        hotel_data = hotel_recs[0].get("hotel", {})
        if "price_per_night" in hotel_data:
            from datetime import datetime
            start = datetime.strptime(state["start_date"], "%Y-%m-%d")
            end = datetime.strptime(state["end_date"], "%Y-%m-%d")
            nights = (end - start).days
            hotel_cost = hotel_data["price_per_night"] * nights
            print(f"🏨  酒店费用: {hotel_data['price_per_night']} x {nights}晚 = {hotel_cost}")

    # 活动费用 - 修正数据访问路径
    activity_cost = 0
    if activity_recs:
        for activity_rec in activity_recs:
            activity_data = activity_rec.get("activity", {})
            if "price" in activity_data:
                cost = activity_data["price"] * state["num_travelers"]
                activity_cost += cost
                print(f"🎉  活动费用: {activity_data['name']} - {activity_data['price']} x {state['num_travelers']} = {cost}")

    # 合计
    total = flight_cost + hotel_cost + activity_cost
    print(f"💰  总费用: {total}")

    breakdown = {
        "flight": flight_cost,
        "hotel": hotel_cost,
        "activities": activity_cost,
        "total": total
    }

    return {
        "budget_breakdown": breakdown,
        "total_cost": total
    }

def final_plan_node(state: TravelPlanState) -> dict:
    """生成最终计划"""
    import json

    prompt = f"""
生成一份详细的旅行计划：

目的地: {state['destination']}
日期: {state['start_date']} 至 {state['end_date']}
人数: {state['num_travelers']}

航班: {json.dumps(state['flight_recommendations'][0], ensure_ascii=False)}
酒店: {json.dumps(state['hotel_recommendations'][0], ensure_ascii=False)}
活动: {json.dumps(state['activity_recommendations'], ensure_ascii=False)}
预算分解: {json.dumps(state['budget_breakdown'], ensure_ascii=False)}

请生成一份清晰、详细的旅行计划。
"""
    response = model.invoke([HumanMessage(content=prompt)])

    return {"final_plan": response.content}

# 添加一个汇聚节点来解决多次执行问题
def gather_expert_results(state: TravelPlanState) -> dict:
    """汇聚所有专家结果"""
    print("📋  汇聚专家推荐结果...")
    
    # 检查所有专家是否都完成了
    flight_done = bool(state.get("flight_recommendations"))
    hotel_done = bool(state.get("hotel_recommendations"))
    activity_done = bool(state.get("activity_recommendations"))
    
    print(f"✈️  航班专家: {'✅' if flight_done else '⏳'}")
    print(f"🏨  酒店专家: {'✅' if hotel_done else '⏳'}")
    print(f"🎉  活动专家: {'✅' if activity_done else '⏳'}")
    
    if flight_done and hotel_done and activity_done:
        print("🎯  所有专家推荐完成，准备计算预算")
        return {"experts_completed": True}
    else:
        print("⏳  等待专家完成...")
        return {"experts_completed": False}

# "final Agent" - 修复版本
main_graph = StateGraph(TravelPlanState)

main_graph.add_node("call_flight_expert", call_flight_expert)
main_graph.add_node("call_hotel_expert", call_hotel_expert)
main_graph.add_node("call_activity_expert", call_activity_expert)
main_graph.add_node("gather_expert_results", gather_expert_results)  # 新增汇聚节点
main_graph.add_node("calculate_budget", calculate_budget)
main_graph.add_node("final_plan", final_plan_node)

# 并行调用三个专家
main_graph.add_edge(START, "call_flight_expert")
main_graph.add_edge(START, "call_hotel_expert")
main_graph.add_edge(START, "call_activity_expert")

# 所有专家完成后先汇聚结果
main_graph.add_edge("call_flight_expert", "gather_expert_results")
main_graph.add_edge("call_hotel_expert", "gather_expert_results")
main_graph.add_edge("call_activity_expert", "gather_expert_results")

# 汇聚完成后计算预算
main_graph.add_edge("gather_expert_results", "calculate_budget")

# 预算计算完成后生成最终计划
main_graph.add_edge("calculate_budget", "final_plan")

# 最终计划完成
main_graph.add_edge("final_plan", END)

# 编译主图
travel_planner = main_graph.compile()

print("🎯  旅行规划系统已就绪！")

🎯  旅行规划系统已就绪！


In [ ]:
# 测试修复后的旅行规划系统
print("=== 测试旅行规划系统 ===")

# 测试输入
test_input = {
    "destination": "北京",
    "start_date": "2024-05-01",
    "end_date": "2024-05-03",
    "budget": 5000.0,
    "num_travelers": 2
}

print(f"📝  测试输入:")
print(f"   目的地: {test_input['destination']}")
print(f"   日期: {test_input['start_date']} 到 {test_input['end_date']}")
print(f"   人数: {test_input['num_travelers']}人")
print(f"   预算: ¥{test_input['budget']}")

print("\n🚀  开始执行旅行规划...")

try:
    # 执行旅行规划
    result = travel_planner.invoke(test_input)
    print("\n✅  旅行规划完成！")
    print("\n📊  预算分解:")
    if "budget_breakdown" in result:
        breakdown = result["budget_breakdown"]
        print(f"   ✈️  航班费用: ¥{breakdown.get('flight', 0)}")
        print(f"   🏨  酒店费用: ¥{breakdown.get('hotel', 0)}")
        print(f"   🎉  活动费用: ¥{breakdown.get('activities', 0)}")
        print(f"   💰  总费用: ¥{breakdown.get('total', 0)}")
        
        # 预算对比
        total_cost = breakdown.get('total', 0)
        budget = test_input['budget']
        if total_cost <= budget:
            print(f"   ✅  在预算范围内 (剩余: ¥{budget - total_cost})")
        else:
            print(f"   ⚠️  超出预算 (超出: ¥{total_cost - budget})")
    
    print(f"\n📋  最终计划:")
    if "final_plan" in result:
        print(result["final_plan"])
    
except Exception as e:
    print(f"❌  执行出错: {e}")
    import traceback
    traceback.print_exc()

print("\n=== 测试完成 ===")

=== 测试旅行规划系统 ===
📝  测试输入:
   目的地: 北京
   日期: 2024-05-01 到 2024-05-03
   人数: 2人
   预算: ¥5000.0

🚀  开始执行旅行规划...
🎉  搜索活动...
🎉  AI 分析活动...
✈️  搜索航班...
✈️  AI 分析航班...
🏨  搜索酒店...
🏨  AI 分析酒店...
📋  汇聚专家推荐结果...
✈️  航班专家: ✅
🏨  酒店专家: ✅
🎉  活动专家: ✅
🎯  所有专家推荐完成，准备计算预算
💰  计算预算...
🔢  开始计算预算...
✈️  航班费用: 950 x 2 = 1900
🏨  酒店费用: 300 x 2晚 = 600
🎉  活动费用: 城市观光 - 200 x 2 = 400
💰  总费用: 2900

✅  旅行规划完成！

📊  预算分解:
   ✈️  航班费用: ¥1900
   🏨  酒店费用: ¥600
   🎉  活动费用: ¥400
   💰  总费用: ¥2900
   ✅  在预算范围内 (剩余: ¥2100.0)

📋  最终计划:
### 旅行计划：北京三日游

**目的地**: 北京  
**旅行日期**: 2024年5月1日至2024年5月3日  
**旅行人数**: 2人  

---

#### 一、航班安排

- **航空公司**: 航空公司 B  
- **出发时间**: 2024年5月1日 14:00  
- **到达时间**: 2024年5月1日 18:00  
- **飞行时长**: 4小时  
- **票价**: 950元/人  
- **总票价**: 1900元（2人）  

**推荐理由**:
1. **价格优势**: 航空公司 B 的票价为950元，较航空公司 A 的1200元更具性价比。
2. **相同的飞行时长**: 两家公司飞行时长相同，均为4小时。
3. **合理的出发时间**: 下午14:00的航班避免了清晨起床的疲劳，适合不喜欢早起的旅客。

---

#### 二、酒店安排

- **酒店名称**: 豪华酒店  
- **星级评分**: 5星  
- **每晚价格**: 800元  
- **住宿时长**: 3晚  
- **总费用**: 2400元（3晚）  

- **设